Основная идея понижения размерности - сохранить расстояния между точками при переходе к меньшей размерности. То есть точки, близкие (далёкие) в пространстве высокой размерности, остаются близкими (далёкими) в пространстве меньшей размерности.

**PCA** (Principal Component Analysis, метод главных компонент) — это способ сжать данные, сохранив в них как можно больше полезной информации. Этот алгоритм находит новые оси/признаки, которые лучше всего описывают разброс данных и оставляет только самые важные. 

PCA строит новые признаки:
- PC1 — направление, где данные различаются сильнее всего
- PC2 — второе по важности направление, перпендикулярное первому
- PC3 — третье, и т.д.

Это новые оси пространства, полученные как линейные комбинации исходных признаков.

**Зачем нужен PCA**
1. Уменьшение размерности. Было 100 признаков, а после стало 5–10.
2. Визуализация данных, то есть можно спроецировать данные в 2D/3D для графиков.
3. Удаление шума.
4. Ускорение ML моделей. Если меньше признаков, то быстрее обучение.

**Минусы**:
- Утрачивается интерпретируемость данных, то есть после применения алгоритма признаки становятся абстрактными.
- Алгоритм работает только с линейными зависимостями. Если структура данных сложная/нелинейная — PCA может плохо помочь.


In [1]:
# Блок с используемыми библиотеками
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
# Блок с описанием и демонстрацией
df = pd.read_csv("../datasets/spotify_tracks.csv", sep=",")
df.head()

,track_id,track_name,artist_name,album_name,release_year,genre,popularity,duration_ms,explicit,danceability,...,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,key,mode,time_signature
0,P3fAbnFbmOHnKYaXRvj7uf,One Dance (Acoustic Version),Alex Rodriguez,The Night Album,2024,metal,14,189042,True,0.427723,...,-4.702460,0.050635,0.239506,0.181395,0.133053,0.431384,141.048735,6,0,4
1,M2wleOV911xCZkwPRQeNHp,Forever Song (Remix),Desert Wind,Burning Soul,2019,rock,11,186805,True,0.448634,...,-7.110031,0.000000,0.044463,0.097818,0.435949,0.559135,131.833287,0,1,5
2,4JSnE2NiiUHUAKw9iEU1jj,Last Mountain,The Midnight,The Night Album,2022,k-pop,23,121814,False,0.707923,...,-7.305120,0.144091,0.118380,0.000000,0.262254,0.516873,127.132954,2,1,5
3,2UVvsjaSS8VFgM0Fmxk754,Falling Star (Live),Phantom Keys,Phantom's Greatest Hits,2024,latin,34,216049,False,0.846237,...,-9.527256,0.006668,0.272844,0.000000,0.045332,0.667911,93.041715,0,1,6
4,EeVVhDIq2AnHTmt9OBGhnu,Rising Moon (feat. someone),Desert Wind,Rising Soul,2010,latin,31,156170,False,0.943720,...,-9.017653,0.057632,0.219752,0.098044,0.132083,0.772151,93.404975,7,1,4


In [3]:
# заменим жанры на категории

df_genre = df.copy()
df_genre = pd.get_dummies(df, columns=['genre'], drop_first=True)

df_tr = df_genre
df_tr.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 39 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   track_id             50000 non-null  str    
 1   track_name           50000 non-null  str    
 2   artist_name          50000 non-null  str    
 3   album_name           50000 non-null  str    
 4   release_year         50000 non-null  int64  
 5   popularity           50000 non-null  int64  
 6   duration_ms          50000 non-null  int64  
 7   explicit             50000 non-null  bool   
 8   danceability         50000 non-null  float64
 9   energy               50000 non-null  float64
 10  loudness             50000 non-null  float64
 11  speechiness          50000 non-null  float64
 12  acousticness         50000 non-null  float64
 13  instrumentalness     50000 non-null  float64
 14  liveness             50000 non-null  float64
 15  valence              50000 non-null  float64
 1

In [4]:
X = df_tr.drop(['track_id', 'track_name', 'artist_name', 'album_name', 'release_year', 'time_signature', 'duration_ms', 'popularity'], 
               axis=1) 

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [5]:
pca = PCA(n_components=10)
pca_result = pca.fit_transform(X_scaled)

## K-means на исходном датасете

In [13]:
kmeans = KMeans(n_clusters=3, random_state=6769)
clusters = kmeans.fit_predict(X_scaled)

In [7]:
score = silhouette_score(X_scaled, clusters)
print("Silhouette Score:", score)

Silhouette Score: 0.1483967376562336


## K-means на датасете с пониженной размерностью

In [16]:
kmeans1 = KMeans(n_clusters=3, random_state=6769)
clusters1 = kmeans1.fit_predict(pca_result)

In [18]:
score = silhouette_score(pca_result, clusters)
print("Silhouette Score:", score)

Silhouette Score: 0.2472492384482212


Видим, что оценка улучшалась. K-means использует расстояния между точками и лишние/коррелированные признаки мешают. Уменьшив размерноть и выбрав оси, мы помогли алгоритму.